<a href="https://colab.research.google.com/github/juanaristud-rgb/Module-4-Ejercicio-4/blob/main/module_4_test_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Generative AI Application with Gradio

In [ ]:
# Install necessary libraries
!pip install -q -U google-generativeai gradio transformers

### API Key Configuration
To use the Gemini API, you'll need an API key. If you don't already have one, create a key in [Google AI Studio](https://makersuite.google.com/app/apikey).

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [ ]:
import google.generativeai as genai
import gradio as gr
from transformers import pipeline
from google.colab import userdata

# Configure Gemini API
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
    print(f"Error configuring Google API key: {e}")
    print("Please ensure 'GOOGLE_API_KEY' is set in Colab secrets.")
    GOOGLE_API_KEY = None # Set to None if API key is not found

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### Initialize Models and Pipelines

In [ ]:
# Initialize Gemini model
try:
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
except Exception as e:
    print(f"Error initializing Gemini model: {e}")
    gemini_model = None
# Initialize Hugging Face sentiment analysis pipeline
try:
    sentiment_pipeline = pipeline("sentiment-analysis")
except Exception as e:
    print(f"Error initializing sentiment analysis pipeline: {e}")
    sentiment_pipeline = None

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### AI Operations

In [ ]:
def detect_and_correct_errors(text):
    if not gemini_model or not GOOGLE_API_KEY:
        return "Gemini model not initialized. Please check your API key and internet connection."
    if not text.strip():
        return "Please enter text to detect and correct errors."
    try:
        prompt = (
            "Analiza el siguiente texto para detectar posibles errores, inconsistencias o afirmaciones dudosas. "
            "Luego, genera una versión corregida o más precisa del texto. Si no encuentras errores, indícalo."
            "\n\nTexto original:\n" + text + "\n\nAnálisis y Corrección:"
        )
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error al procesar con Gemini: {e}"

def analyze_sentiment(text):
    if not sentiment_pipeline:
        return "Hugging Face sentiment analysis pipeline not initialized."
    if not text.strip():
        return "Please enter text for sentiment analysis."
    try:
        result = sentiment_pipeline(text)
        label = result[0]['label']
        score = result[0]['score']

        color = "black" # Default color for neutral or if sentiment cannot be determined
        if label == 'POSITIVE':
            color = "green"
        elif label == 'NEGATIVE':
            color = "red"

        # Return HTML formatted string for colored output
        return f"<p style=\"color:{color}; font-weight:bold;\">Sentimiento: {label} (Confianza: {score:.2f})</p>"
    except Exception as e:
        return f"Error al analizar el sentimiento: {e}"

def rephrase_text(text):
    if not gemini_model or not GOOGLE_API_KEY:
        return "Gemini model not initialized. Please check your API key and internet connection."
    if not text.strip():
        return "Please enter text to rephrase."
    try:
        prompt = (
            "Reescribe el siguiente texto de forma más clara y sencilla, manteniendo el mismo significado. "
            "\n\nTexto original:\n" + text + "\n\nTexto reformulado:"
        )
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error al reformular con Gemini: {e}"

In [ ]:
print("Checking for available generative models...")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

Checking for available generative models...
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-m

Please run the above cell to see the list of available models. Based on the output, we will update the `gemini_model` initialization in cell `66ffb892` to use an appropriate model (e.g., `gemini-1.0-pro` or `gemini-1.5-pro-latest` if available). If `gemini-pro` is listed, there might be a temporary API issue, and trying another similar model is advisable.

After running the cell above, you will need to manually edit cell `66ffb892` to replace `gemini-pro` with one of the models listed. Common choices include `gemini-1.0-pro` or `gemini-1.5-pro-latest`.

Then, rerun cells `66ffb892`, `aa3077cb`, and `147425f8`.

### Gradio Interface

In [ ]:
import gradio as gr

# Define a custom Gradio Theme for a more unique look
# Using Base() to have more control, then customize colors and fonts
# Gr.Theme helps set base colors for components, while custom_css handles layout and advanced styling
custom_theme = gr.Theme(
    primary_hue="blue",  # Main interactive elements like active tabs, primary buttons
    secondary_hue="green",  # Secondary interactive elements or accents
    neutral_hue="gray",    # Backgrounds and borders for non-interactive elements
    text_size="lg",        # Set base text size
).set(
    # General background. Note: body_background_fill from custom_css will likely override this
    background_fill_primary='rgba(240, 240, 240, 0.95)',
    background_fill_secondary='rgba(230, 230, 230, 0.7)',

    # Button styling (primary and secondary)
    button_primary_background_fill_dark='linear-gradient(90deg, #1A2980, #26D0CE)', # Dark Blue to Cyan
    button_primary_background_fill_hover_dark='linear-gradient(90deg, #26D0CE, #1A2980)',
    button_primary_background_fill='linear-gradient(90deg, #1A2980, #26D0CE)',
    button_primary_background_fill_hover='linear-gradient(90deg, #26D0CE, #1A2980)',
    button_primary_text_color='white',
    button_primary_border_color='transparent',

    button_secondary_background_fill_dark='linear-gradient(90deg, #0F2027, #203A43)', # Dark Slate Blue to Dark Teal
    button_secondary_background_fill_hover_dark='linear-gradient(90deg, #203A43, #0F2027)',
    button_secondary_background_fill='linear-gradient(90deg, #0F2027, #203A43)',
    button_secondary_background_fill_hover='linear-gradient(90deg, #203A43, #0F2027)',
    button_secondary_text_color='white',
    button_secondary_border_color='transparent',

    # Input components like Textbox
    input_background_fill='rgba(255, 255, 255, 0.95)',
    input_border_color_focus='#1A2980',
    input_border_width='2px',
    input_shadow='0 2px 5px rgba(0,0,0,0.1)',

    # Container and block styling
    block_background_fill='transparent',
    container_radius='20px',
    block_border_width='0px',
    shadow_drop_lg='0 10px 20px rgba(0, 0, 0, 0.2)' # Better shadow for blocks
)

# More specific and forceful custom CSS for advanced styling and overrides
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Pacifico&family=Inter:wght@400;700&display=swap');

html, body {
    background: linear-gradient(135deg, #0F2027, #203A43, #2C5364) !important; /* Dark Slate Blue to Teal gradient */
    font-family: 'Inter', sans-serif !important;
    color: #E0E0E0 !important; /* Light gray text for contrast */
}
.gradio-container {
    background-color: rgba(255, 255, 255, 0.98) !important; /* Almost opaque white background for content */
    border-radius: 25px !important;
    box-shadow: 0 15px 30px rgba(0, 0, 0, 0.5) !important;
    padding: 40px !important;
    margin-top: 30px !important;
    margin-bottom: 30px !important;
}
h1 {
    color: #1A2980 !important; /* Dark Blue */
    text-align: center !important;
    font-family: 'Inter', sans-serif !important; /* Changed from Pacifico to Inter */
    font-size: 3.8em !important;
    margin-bottom: 30px !important;
    text-shadow: 3px 3px 6px rgba(0,0,0,0.4);
    letter-spacing: 2px;
}
h2 {
    color: #2C5364 !important; /* Dark Teal */
    font-family: 'Inter', sans-serif !important;
    font-weight: 700 !important;
    font-size: 2.2em !important;
    border-bottom: 4px solid #26D0CE !important; /* Cyan border */
    padding-bottom: 10px !important;
    margin-top: 35px !important;
    text-align: left !important;
}
.gr-button {
    border-radius: 30px !important;
    font-weight: bold !important;
    font-size: 1.25em !important;
    padding: 14px 28px !important;
    transition: all 0.3s ease-in-out !important;
    box-shadow: 0 5px 10px rgba(0, 0, 0, 0.35);
    text-transform: uppercase;
}
.gr-button:hover {
    transform: translateY(-4px) scale(1.03) !important;
    box-shadow: 0 8px 16px rgba(0, 0, 0, 0.45) !important;
    opacity: 0.9;
}
.gr-tab-button {
    font-size: 1.4em !important;
    font-weight: 700 !important;
    padding: 18px 30px !important;
    border-radius: 20px 20px 0 0 !important;
    box-shadow: 0 -8px 15px rgba(0, 0, 0, 0.2) !important;
    color: #34495E !important; /* Dark Grey-Blue */
    background-color: #ECEFF1 !important; /* Light Blue Grey */
    transition: all 0.3s ease;
}
.gr-tab-button.selected {
    color: white !important;
    background-color: #007BFF !important; /* Vibrant Blue */
    box-shadow: 0 -10px 20px rgba(0, 0, 0, 0.4) !important;
}
.gr-textbox label, .gr-markdown h2 {
    color: #1A2980 !important; /* Dark Blue for labels */
    font-weight: bold !important;
    font-size: 1.1em !important;
}
/* Custom style for the colored sentiment output */
.gr-markdown p {
    font-size: 1.3em !important;
    line-height: 1.6em !important;
    padding: 10px 0;
    color: #333 !important; /* Ensure readable text color inside markdown output */
}
.gradio-app {
    min-height: 100vh !important; /* Ensure app takes full viewport height */
    display: flex;
    justify-content: center;
    align-items: center;
}
"""

with gr.Blocks(title="Generative AI Assistant", theme=custom_theme, css=custom_css) as demo:
    gr.Markdown("# ✨ Asistente de IA Generativa Creativa y Única ✨")

    with gr.Tab("📝 Detector y Corrector de Errores"):
        gr.Markdown("## Detector y Corrector de Errores/Mentiras con Gemini")
        error_input = gr.Textbox(label="Pega tu texto aquí para análisis", lines=10, placeholder="Ejemplo: 'El sol gira alrededor de la tierra.' Esta afirmación contiene un error científico.")
        error_output = gr.Textbox(label="Análisis y Corrección Detallada", lines=10, interactive=False)
        error_button = gr.Button("🔍 Analizar y Corregir Errores")
        error_button.click(detect_and_correct_errors, inputs=error_input, outputs=error_output)

    with gr.Tab("⭐ Analizador de Reseñas"):
        gr.Markdown("## Análisis de Sentimiento con HuggingFace")
        sentiment_input = gr.Textbox(label="Introduce tu reseña o comentario", lines=5, placeholder="Ejemplo: '¡Me encanta este producto, es fantástico! Ha superado todas mis expectativas.'")
        sentiment_output = gr.Markdown(label="Resultado del Sentimiento")
        sentiment_button = gr.Button("😊 Analizar Sentimiento")
        sentiment_button.click(analyze_sentiment, inputs=sentiment_input, outputs=sentiment_output)

    with gr.Tab("✍️ Reorganizador y Corrector de Textos"):
        gr.Markdown("## Reorganizador y Corrector de Textos con Gemini")
        rephrase_input = gr.Textbox(label="Pega el texto a reorganizar/corregir", lines=10, placeholder="Ejemplo: 'La complejidad intrínseca de los sistemas cuánticos demanda una aproximación metodológica rigurosa, lo que a menudo dificulta su comprensión para el público general.'")
        rephrase_output = gr.Textbox(label="Texto Reorganizado y Más Claro", lines=10, interactive=False)
        rephrase_button = gr.Button("✏️ Reorganizar y Corregir")
        rephrase_button.click(rephrase_text, inputs=rephrase_input, outputs=rephrase_output)

demo.launch(debug=True)

/tmp/ipykernel_12371/3069378641.py:129: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Generative AI Assistant", theme=custom_theme, css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95582c9c4f0534f1e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
